# Predictive AI Evaluation Challenge: A Practitioner's Iteration Log

This notebook is the companion reproducibility artifact for the Stanford CS321M Predictive AI Evaluation Challenge entry that scored **NLL = -0.59 (tied #6 of 14)** on the Codabench leaderboard. It walks through:

1. The cold-start prediction setting and what makes it hard
2. `ColdStartLookupPredictor` -- the six-level hierarchical empirical-Bayes model deployed as the final submission
3. The intercept-only Platt calibration mechanism and why it differs from full Platt
4. `LLMJudgeIRT` -- the 1-PL IRT augmentation that was *not* deployed and why
5. A reproducible negative-result analysis on synthetic data

**Why this notebook exists.** The competition leaderboard rewards predictions, but the technical-report and GitHub artifacts are graded on engineering discipline and reproducibility. This tutorial demonstrates both models in self-contained CPU-only Python (no HuggingFace download, no GPU) so a reviewer can verify every numerical claim in the report by re-running cells in order.

**Runtime:** ~30 seconds on CPU.

## 1. The cold-start setting

Let $X_{ij} \in \{0, 1\}$ denote whether AI model $i$ correctly answers item $j$. At training time we observe $X$ on a dense matrix of 909 subjects $\times$ ~104K items. At test time we get *new* items but the *same* subjects, plus four fields per row: a benchmark id, a condition, a multi-line subject_content block, and the raw item text.

This is the **item cold-start** regime. The relevant axis of variation is item difficulty, not subject ability -- the subjects are known and their priors can be learned. The competition supplies $K=5$ ground-truth labels per benchmark via an adaptive labelling channel before `predict()` is called.

Metric: mean log-likelihood. Higher is better, $\mathcal{L} = 0$ is the upper bound, $\hat{p} = 0.5$ uniformly gives $\mathcal{L} = -\log 2 \approx -0.693$.

In [1]:
import math

from torch_measure.models import ColdStartLookupPredictor, LLMJudgeIRT, build_difficulty_prompt

/Users/vasundras/anaconda3/envs/torch_measure/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 2. The deployed model: `ColdStartLookupPredictor`

The model walks a six-level fallback hierarchy and returns the first match. Each level is tried exact, then case-insensitive.

| Level | Key                                          | Source                          |
|-------|----------------------------------------------|---------------------------------|
| 1     | `(subject, benchmark, condition)`            | training triples, $n \geq 3$    |
| 2     | `(subject, benchmark)`                       | Bayesian-shrunk, $n \geq 1$     |
| 3     | $\sigma(\mathrm{logit}\,s + \mathrm{logit}\,b - \mathrm{logit}\,\mu)$ | 1-PL Rasch IRT blend |
| 4     | benchmark prior                              | unknown subject                 |
| 5     | subject prior                                | unknown benchmark               |
| 6     | global mean                                  | both unknown                    |

Let's build a tiny synthetic lookup that mirrors the structure of the real one (built from 5.36M training responses):

In [2]:
predictor = ColdStartLookupPredictor(
    sbc={
        'gpt-4||mmlupro||zero-shot': 0.78,
        'gpt-4||mmlupro||cot':       0.85,
        'claude-3||ai2d_test||none': 0.72,
    },
    sb={
        'gpt-4||mmlupro':      0.80,
        'gpt-4||cybench':      0.40,
        'claude-3||ai2d_test': 0.72,
        'llama-2-7b||cybench': 0.18,
    },
    subj={'gpt-4': 0.75, 'claude-3': 0.70, 'llama-2-7b': 0.35},
    bench={'mmlupro': 0.50, 'cybench': 0.25, 'ai2d_test': 0.70},
    global_mean=0.645,
    name_aliases={'Llama-2-7b-chat': 'llama-2-7b'},
    name_lc={'gpt-4': 'gpt-4', 'claude-3': 'claude-3', 'llama-2-7b': 'llama-2-7b'},
)
predictor

### 2.1 Exercising each fallback level

Below we construct test records that deliberately miss each level until we reach the desired fallback, demonstrating the cascading hierarchy:

In [3]:
def show(level_name, record, expected_source):
    p = predictor.predict(record)
    print(f'{level_name:8s} | benchmark={record["benchmark"]:10s} cond={record["condition"]:10s} '
          f'subj={record["subject_content"][6:30]:25s} -> P={p:.4f}  ({expected_source})')

show('Level 1', {'benchmark': 'mmlupro',  'condition': 'cot',  'subject_content': 'Name: gpt-4',
                 'item_content': 'q'},   'triple match')
show('Level 2', {'benchmark': 'mmlupro',  'condition': 'few-shot', 'subject_content': 'Name: gpt-4',
                 'item_content': 'q'},   '(subj, bench) match, unseen condition')
show('Level 3', {'benchmark': 'ai2d_test','condition': 'none', 'subject_content': 'Name: gpt-4',
                 'item_content': 'q'},   'IRT blend (subj and bench priors both known)')
show('Level 4', {'benchmark': 'mmlupro',  'condition': 'none', 'subject_content': 'Name: NewModel-X',
                 'item_content': 'q'},   'benchmark prior only')
show('Level 5', {'benchmark': 'new_bench','condition': 'none', 'subject_content': 'Name: gpt-4',
                 'item_content': 'q'},   'subject prior only')
show('Level 6', {'benchmark': 'new_bench','condition': 'none', 'subject_content': 'Name: NewModel-X',
                 'item_content': 'q'},   'global fallback')

Level 1  | benchmark=mmlupro    cond=cot        subj=gpt-4                     -> P=0.8500  (triple match)
Level 2  | benchmark=mmlupro    cond=few-shot   subj=gpt-4                     -> P=0.8000  ((subj, bench) match, unseen condition)
Level 3  | benchmark=ai2d_test  cond=none       subj=gpt-4                     -> P=0.7939  (IRT blend (subj and bench priors both known))
Level 4  | benchmark=mmlupro    cond=none       subj=NewModel-X                -> P=0.5000  (benchmark prior only)
Level 5  | benchmark=new_bench  cond=none       subj=gpt-4                     -> P=0.7500  (subject prior only)
Level 6  | benchmark=new_bench  cond=none       subj=NewModel-X                -> P=0.6450  (global fallback)


### 2.2 Closed-form check of the level-3 IRT blend

Level 3 implements the classical Rasch model expressed against the global mean:

$$
P(\text{correct} \mid \text{subj}, \text{bench}) \;=\; \sigma\!\bigl(\mathrm{logit}\,s \;+\; \mathrm{logit}\,b \;-\; \mathrm{logit}\,\mu\bigr).
$$

For the gpt-4 $\times$ ai2d_test pair above ($s=0.75,\, b=0.70,\, \mu=0.645$), the expected value is computable by hand:

In [4]:
def logit(p): return math.log(p / (1 - p))
def sigmoid(x): return 1 / (1 + math.exp(-x))

expected = sigmoid(logit(0.75) + logit(0.70) - logit(0.645))
actual   = predictor.predict({'benchmark': 'ai2d_test', 'condition': 'none',
                              'subject_content': 'Name: gpt-4', 'item_content': 'q'})
print(f'Closed-form IRT blend: {expected:.6f}')
print(f'Predictor output:      {actual:.6f}')
print(f'Match within tolerance: {abs(expected - actual) < 1e-6}')

Closed-form IRT blend: 0.793930
Predictor output:      0.793930
Match within tolerance: True


### 2.3 Name resolution: provider prefixes

A meaningful failure mode for the earlier M2 submission was that `subject_content` at test time often arrives with provider-prefixed names (e.g., `meta-llama/Llama-2-7b-chat`) that do not match the bare `display_name` stored in training (`Llama-2-7b-chat`). `ColdStartLookupPredictor` strips fourteen known prefixes and consults an alias map:

In [5]:
# Two representations of the same subject -- both must hit the same level-2 entry.
p_canonical = predictor.predict({'benchmark': 'cybench', 'condition': 'none',
                                  'subject_content': 'Name: llama-2-7b', 'item_content': 'q'})
p_prefixed  = predictor.predict({'benchmark': 'cybench', 'condition': 'none',
                                  'subject_content': 'Name: meta-llama/Llama-2-7b-chat', 'item_content': 'q'})
print(f'Canonical name: {p_canonical:.4f}')
print(f'Prefixed name:  {p_prefixed:.4f}')
print(f'Match: {abs(p_canonical - p_prefixed) < 1e-6}')

Canonical name: 0.1800
Prefixed name:  0.1800
Match: True


### 2.4 Adaptive calibration with `K=5` revealed labels

Before `predict()` is called, the competition reveals $K=5$ ground-truth labels per benchmark. We fit an *intercept-only* Platt scaler: slope is fixed at 1, only the shift $b$ is fit, capped at $\pm 1.5$ logit units. The reason: full Platt at $K=5$ produces a near-flat calibrator ($a \approx 0.1$) that destroys subject ordering. Locking $a=1$ preserves the ordering and only shifts the round's predictions to match the empirical label mean.

Demonstration: a 4-out-of-5 zero-label batch on mmlupro should pull mmlupro predictions down. cybench predictions must be unchanged.

In [6]:
labeled = [
    {'benchmark': 'mmlupro', 'condition': 'zero-shot', 'subject_content': 'Name: gpt-4',      'label': 0},
    {'benchmark': 'mmlupro', 'condition': 'zero-shot', 'subject_content': 'Name: claude-3',   'label': 0},
    {'benchmark': 'mmlupro', 'condition': 'zero-shot', 'subject_content': 'Name: llama-2-7b', 'label': 0},
    {'benchmark': 'mmlupro', 'condition': 'zero-shot', 'subject_content': 'Name: gpt-4',      'label': 0},
    {'benchmark': 'mmlupro', 'condition': 'zero-shot', 'subject_content': 'Name: claude-3',   'label': 1},
]

mm_record = {'benchmark': 'mmlupro', 'condition': 'zero-shot',
             'subject_content': 'Name: gpt-4', 'item_content': 'q'}
cy_record = {'benchmark': 'cybench', 'condition': 'none',
             'subject_content': 'Name: gpt-4', 'item_content': 'q'}

print('              raw      calibrated   shift')
for name, rec in [('mmlupro', mm_record), ('cybench', cy_record)]:
    raw = predictor.predict(rec)
    cal = predictor.predict(rec, labeled)
    print(f'{name:8s}      {raw:.4f}   {cal:.4f}      {cal - raw:+.4f}')

print('\nObserve: mmlupro shifted DOWN (4/5 zero labels); cybench unchanged (no calibration data).')

              raw      calibrated   shift
mmlupro       0.7800   0.4417      -0.3383
cybench       0.4000   0.4000      +0.0000

Observe: mmlupro shifted DOWN (4/5 zero labels); cybench unchanged (no calibration data).


## 3. The experimental model: `LLMJudgeIRT`

The lookup model treats all items inside a (subject, benchmark, condition) cell as exchangeable, which they are not. The M4.5 architecture probes whether a frozen LLM judge can extract per-item difficulty signal to augment the lookup prior:

$$
\hat p \;=\; \sigma\!\bigl(\theta_{\text{subject}} \;-\; \delta_{\text{item}}\bigr),
$$

where $\theta_{\text{subject}} = \mathrm{logit}\,\hat p_{\text{lookup}}$ is the M3 prior in logit space, and $\delta_{\text{item}} = \alpha (\log P_{\text{yes}} - \log P_{\text{no}}) + \beta$ comes from a 14B instruction-tuned LLM judge given a difficulty-rating prompt.

Here we use a mock judge (deterministic, no GPU) to demonstrate the IRT combination math. The competition build uses Qwen2.5-14B-Instruct.

In [7]:
# Inspect the difficulty prompt format. Note the prompt deliberately does NOT mention
# the subject -- this is by design (subject behaviour prediction failed at 7B scale).
print(build_difficulty_prompt(item_content='What is the integral of e^x?', benchmark='mmlupro'))

Below is a question from the benchmark `mmlupro`. Without solving it, rate whether this question would be difficult for a typical AI assistant.

Question:
What is the integral of e^x?

Is this question difficult? Answer with one word, yes or no.
Answer:


In [8]:
# Mock judge: assigns positive logit ("hard") to long items, negative ("easy") to short ones.
# This is a stand-in; the real judge uses 14B-Instruct next-token logits at the 'Answer:' position.
def mock_judge(item_content, benchmark):
    return (len(item_content) - 100) / 50.0  # signed, magnitude grows with item length

model = LLMJudgeIRT(lookup=predictor, judge_fn=mock_judge, alpha=0.20, beta=0.0)

for label, ic in [('short item (easy)', 'q'),
                  ('medium item',       'q' * 100),
                  ('long item (hard)',  'q' * 500)]:
    rec = {'benchmark': 'mmlupro', 'condition': 'zero-shot',
           'subject_content': 'Name: gpt-4', 'item_content': ic}
    p_lookup = predictor.predict(rec)
    p_irt    = model.predict(rec)
    print(f'{label:25s} P_lookup={p_lookup:.4f}  P_IRT={p_irt:.4f}  shift={p_irt - p_lookup:+.4f}')

print('\nIRT prediction shifts down for longer (harder, per the mock judge) items.')

short item (easy)         P_lookup=0.7800  P_IRT=0.8405  shift=+0.0605
medium item               P_lookup=0.7800  P_IRT=0.7800  shift=+0.0000
long item (hard)          P_lookup=0.7800  P_IRT=0.4172  shift=-0.3628

IRT prediction shifts down for longer (harder, per the mock judge) items.


### 3.1 Why we did not deploy this model: the convergent negative result

Across three rigorously validated iterations on the real competition data, the LLM-judge signal proved too small to reliably exceed the M3 lookup baseline. The pattern was convergent:

| Iteration | Fit rows | Raw $r$ | Val NLL lift | Smoke NLL lift |
|-----------|----------|---------|--------------|----------------|
| v1 (no validation)    | 600  | $-0.252$ | --        | $-0.024$ |
| v2 (validated $\alpha$) | 600  | $-0.252$ | $+0.009$ | $-0.021$ |
| v3 (1500 rows)        | 1500 | $-0.257$ | $+0.002$ | $-0.006$ |

The judge does extract real signal ($r \approx -0.26$ is stable across sample sizes). But the validation lift collapses to $+0.002$ NLL as the validation protocol tightens, while sample-to-sample noise on a 200-row smoke set is $\approx 0.020$ NLL -- roughly $5\times$ larger than the converged effect.

Below we reproduce this pattern on synthetic data to demonstrate the diagnostic logic. We simulate a regime where the judge has a true effect of $\alpha^\star = 0.05$ and noise variance $\sigma^2 = 1$:

In [9]:
import random
random.seed(42)

TRUE_ALPHA = 0.05
TRUE_BETA  = 0.0

def synthetic_dataset(n_rows):
    """Generate (theta, judge_logit, label) triples where the judge has small true effect."""
    rows = []
    for _ in range(n_rows):
        theta = random.gauss(0.5, 0.5)
        jl    = random.gauss(0.0, 1.0)
        # True P = sigmoid(theta - alpha*jl - beta)
        p_true = 1 / (1 + math.exp(-(theta - TRUE_ALPHA * jl - TRUE_BETA)))
        label = 1 if random.random() < p_true else 0
        rows.append((theta, jl, label))
    return rows

def eval_nll(rows, alpha, beta):
    nll = 0.0
    for theta, jl, y in rows:
        delta = alpha * jl + beta
        z = max(-3.0, min(3.0, theta - delta))
        p = 1 / (1 + math.exp(-z))
        p = max(1e-6, min(1 - 1e-6, p))
        nll += y * math.log(p) + (1 - y) * math.log(1 - p)
    return nll / len(rows)

for n_fit in [600, 1500, 5000]:
    fit   = synthetic_dataset(n_fit)
    smoke = synthetic_dataset(200)

    thetas = [r[0] for r in fit]
    jls    = [r[1] for r in fit]
    labels = [r[2] for r in fit]
    alpha, beta, _ = LLMJudgeIRT.fit_alpha_beta(thetas, jls, labels, l2_penalty=2.0)

    smoke_nll_irt = eval_nll(smoke, alpha, beta)
    smoke_nll_lookup = eval_nll(smoke, 0.0, 0.0)
    print(f'n_fit={n_fit:5d}  fitted alpha={alpha:+.4f}  smoke lift over lookup: {smoke_nll_irt - smoke_nll_lookup:+.4f}')

n_fit=  600  fitted alpha=+0.0097  smoke lift over lookup: +0.0004
n_fit= 1500  fitted alpha=+0.0053  smoke lift over lookup: -0.0003
n_fit= 5000  fitted alpha=+0.0024  smoke lift over lookup: -0.0007


Notice that even with the *true* $\alpha^\star = 0.05$ baked into the data, a 200-row smoke set is too small to reliably reward the model -- the smoke lift swings around zero based on the random draw. This is precisely the regime our LLM-judge experiment landed in.

### 3.2 The methodology contribution

The decision to not deploy M4.5 rests on three pre-registered engineering rules. Each one would have been violated by the v1 iteration:

1. **Smoke-test thresholds must be expressed as differences from the baseline being supplanted, not as absolute values.** v1 used an absolute AUC threshold and a regression slipped through.
2. **Validation-set $\alpha$ tuning with L2 regularisation prevents overfitting** when $K \ll n$. v2 introduced this; the apparent fit-set lift halved.
3. **Composite-key exclusion when expanding training samples** prevents contaminating the held-out smoke set. v3 enforced this on the (subject_id, item_id, benchmark_id, condition) quadruple.

## 4. Running the package unit tests

All deployment-critical invariants are codified in the test suite. From the package root: `pytest tests/test_models/test_cold_start_lookup.py tests/test_models/test_llm_judge_irt.py -v` runs 37 tests in under a second:

In [10]:
import subprocess
result = subprocess.run(
    ['pytest', '../tests/test_models/test_cold_start_lookup.py',
     '../tests/test_models/test_llm_judge_irt.py', '--tb=line', '-q'],
    capture_output=True, text=True,
)
print(result.stdout.split('\n')[-3] if result.stdout else result.stderr.split('\n')[-3])

.....................................                                    [100%]


## 5. Summary

**Deployed:** `ColdStartLookupPredictor` with six-level fallback, Bayesian shrinkage at level 2, 1-PL IRT blend at level 3, intercept-only Platt calibration at runtime. Leaderboard NLL = -0.59 (tied #6).

**Not deployed:** `LLMJudgeIRT` with Qwen2.5-14B-Instruct as the item-difficulty rater. Three rigorous iterations confirmed the judge's effect size is below the sample-to-sample variance threshold for confident deployment. Documented as an experimental model in the package with explicit guidance against production use.

**Practitioner takeaways:**

1. Cold-start item prediction with known subjects is dominated by subject-conditioned benchmark priors. Beating $-0.59$ on the leaderboard requires item-level signal that genuinely transfers across distribution shifts -- a high bar that the 14B judge does not clear on its own.
2. Field-name mismatches and silent name-resolution failures (the M1, M2 bugs) account for most of the gap between $-0.71$ and $-0.59$. Architecture is not the bottleneck below the lookup floor.
3. Pre-registered smoke-test gates expressed as *differences* from a deployed baseline are the engineering safeguard that converts a hopeful submission into a deliberate one. A submission that fails the gate is a successful test, not a wasted iteration.